In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# 경로 설정
DATA_DIR = Path('../Data')
OUTPUT_DIR = Path('../Data')

## 데이터 로드

In [22]:
# all_events 로드
all_events = pd.read_csv(DATA_DIR / 'all_events.csv')
print(f"all_events shape: {all_events.shape}")
print(f"Columns: {all_events.columns.tolist()}")
all_events.head()

C:\Users\김한재\AppData\Local\Temp\ipykernel_31772\2901030276.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  all_events = pd.read_csv(DATA_DIR / 'all_events.csv')


all_events shape: (38276100, 8)
Columns: ['stay_id', 'itemid', 'label', 'value_str', 'value_num', 'valueuom', 'charttime', 'source']


,stay_id,itemid,label,value_str,value_num,valueuom,charttime,source
0,30983975,223761,Temperature Fahrenheit,98.8,98.8,°F,2175-10-21 08:00:00.000,chart
1,30983975,220277,O2 saturation pulseoxymetry,100,100.0,%,2175-10-21 08:00:00.000,chart
2,30983975,228096,Richmond-RAS Scale,"""+2 Frequent nonpurposeful movement",NaN,2,2175-10-21 08:00:00.000,chart
3,30983975,228299,Goal Richmond-RAS Scale,0 Alert and calm,0.0,NaN,2175-10-21 08:00:00.000,chart
4,30983975,220277,O2 saturation pulseoxymetry,100,100.0,%,2175-10-21 09:00:00.000,chart


In [4]:
# adm_pat_icu 로드 (환자 기본 정보)
adm_pat_icu = pd.read_csv(DATA_DIR / 'adm_pat_icu.csv')
print(f"adm_pat_icu shape: {adm_pat_icu.shape}")
print(f"Columns: {adm_pat_icu.columns.tolist()}")
adm_pat_icu.head()

adm_pat_icu shape: (72729, 9)
Columns: ['subject_id', 'hadm_id', 'stay_id', 'gender', 'age', 'los', 'admission_type', 'intime', 'outtime']


,subject_id,hadm_id,stay_id,gender,age,los,admission_type,intime,outtime
0,10000032,29079034,39553978,1,52,0.410266,EW EMER.,2180-07-23 14:00:00.000,2180-07-23 23:50:47.000
1,10000980,26913865,39765666,1,73,0.497535,EW EMER.,2189-06-27 08:42:00.000,2189-06-27 20:38:27.000
2,10001217,24597018,37067082,1,55,1.118032,EW EMER.,2157-11-20 19:18:02.000,2157-11-21 22:08:00.000
3,10001217,27703517,34592300,1,55,0.948113,DIRECT EMER.,2157-12-19 15:42:24.000,2157-12-20 14:27:41.000
4,10001725,25563031,31205490,1,46,1.338588,EW EMER.,2110-04-11 15:52:22.000,2110-04-12 23:59:56.000


In [ ]:
# 날짜 컬럼 변환
all_events['charttime'] = pd.to_datetime(all_events['charttime'])
adm_pat_icu['intime'] = pd.to_datetime(adm_pat_icu['intime'])
if 'outtime' in adm_pat_icu.columns:
    adm_pat_icu['outtime'] = pd.to_datetime(adm_pat_icu['outtime'])

In [ ]:
# LOS(Length of Stay) 8일 미만 데이터 제거
print(f"Before filtering: {len(adm_pat_icu)} stays")
adm_pat_icu = adm_pat_icu[adm_pat_icu['los'] >= 8]
print(f"After filtering (los >= 8): {len(adm_pat_icu)} stays")
adm_pat_icu.to_csv(OUTPUT_DIR / 'adm_pat_icu_8hrs.csv', index=False)

# all_events에서도 해당 stay_id만 유지
valid_stay_ids = set(adm_pat_icu['stay_id'])
all_events = all_events[all_events['stay_id'].isin(valid_stay_ids)]
print(f"all_events after filtering: {all_events.shape}")
all_events.to_csv(OUTPUT_DIR / 'all_events_8hrs.csv', index=False)

In [103]:
adm_pat_icu = pd.read_csv(DATA_DIR / 'adm_pat_icu_8hrs.csv')
all_events = pd.read_csv(DATA_DIR / 'all_events_8hrs.csv')

C:\Users\김한재\AppData\Local\Temp\ipykernel_16860\2787265544.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  all_events = pd.read_csv(DATA_DIR / 'all_events_8hrs.csv')


## STEP 1: VALUE 변환 (문자열 → 숫자)

In [104]:
all_events.head()

,stay_id,itemid,label,value_str,value_num,valueuom,charttime,source
0,30983975,223761,Temperature Fahrenheit,98.8,98.8,°F,2175-10-21 08:00:00,chart
1,30983975,220277,O2 saturation pulseoxymetry,100,100.0,%,2175-10-21 08:00:00,chart
2,30983975,228096,Richmond-RAS Scale,"""+2 Frequent nonpurposeful movement",NaN,2,2175-10-21 08:00:00,chart
3,30983975,228299,Goal Richmond-RAS Scale,0 Alert and calm,0.0,NaN,2175-10-21 08:00:00,chart
4,30983975,220277,O2 saturation pulseoxymetry,100,100.0,%,2175-10-21 09:00:00,chart


In [ ]:
all_events = all_events[all_events.label != 'Bilirubin']

In [106]:
for i in all_events[all_events['value_num'].isnull()].value_str.unique():
    print(i)

"+2 Frequent nonpurposeful movement
"+1 Anxious
Hamilton
"-3 Moderate sedation
"-4 Deep sedation
"-5 Unarousable
"-2 Light sedation
"+4 Combative
No
No (less than 3 errors - Stop - Not delirious)
Yes
"Yes (3 or more errors
BiPAP/CPAP
nan
___
<0.1


In [107]:
all_events.loc[all_events['value_str'] == '"+2 Frequent nonpurposeful movement', 'value_num'] = 2
all_events.loc[all_events['value_str'] == '"+1 Anxious', 'value_num'] = 1
all_events.loc[all_events['value_str'] == '"-3 Moderate sedation', 'value_num'] = -3
all_events.loc[all_events['value_str'] == '"-4 Deep sedation', 'value_num'] = -4
all_events.loc[all_events['value_str'] == '"-5 Unarousable' , 'value_num'] = -5
all_events.loc[all_events['value_str'] == '"-2 Light sedation' , 'value_num'] = -2
all_events.loc[all_events['value_str'] == '"+4 Combative' , 'value_num'] = 4
all_events.loc[all_events['value_str'] == 'No' , 'value_num'] = 0
all_events.loc[all_events['value_str'] == 'No (less than 3 errors - Stop - Not delirious)' , 'value_num'] = 0
all_events.loc[all_events['value_str'] == 'Yes' , 'value_num'] = 1
all_events.loc[all_events['value_str'] == '"Yes (3 or more errors' , 'value_num'] = 1
all_events.loc[all_events['value_str'] == '<0.1' , 'value_num'] = 0

all_events.loc[all_events['label'] == 'Ventilator Type' , 'value_num'] = 1 # 사용여부

In [108]:
all_events.rename(columns={'value_num':'value'}, inplace=True)

## STEP 2: 단위 변환

In [109]:
# 단위 변환

def fahr_to_celsius(temp_fahr):
    """Convert Fahrenheit to Celsius
    Return Celsius conversion of input"""
    temp_celsius = (temp_fahr - 32) * 5 / 9
    return temp_celsius

# Temperature: °F → °C
temp_f_mask = all_events['label'].str.contains('Fahrenheit', case=False, na=False)
print(f"Temperature Fahrenheit rows: {temp_f_mask.sum()}")
print(f"Before conversion - Temperature F describe:\n{all_events.loc[temp_f_mask, 'value'].describe()}\n")
all_events.loc[temp_f_mask, 'value'] = fahr_to_celsius(all_events.loc[temp_f_mask, 'value'])
print(f"After conversion - Temperature (now °C) describe:\n{all_events.loc[temp_f_mask, 'value'].describe()}\n")

# Weight: lbs → kg
weight_lbs_mask = all_events['label'].str.contains('lbs', case=False, na=False)
print(f"Weight lbs rows: {weight_lbs_mask.sum()}")
all_events.loc[weight_lbs_mask, 'value'] = all_events.loc[weight_lbs_mask, 'value'] * 0.453592

# Height: inch → cm
height_inch_mask = (
    all_events['label'].str.contains('inch', case=False, na=False) |
    (all_events['valueuom'].fillna('').str.lower().isin(['inch', 'in']))
)
print(f"Height inch rows: {height_inch_mask.sum()}")
all_events.loc[height_inch_mask, 'value'] = all_events.loc[height_inch_mask, 'value'] * 2.54

print(f"\nUnit conversion done. Shape: {all_events.shape}")

Temperature Fahrenheit rows: 738677
Before conversion - Temperature F describe:
count    738677.000000
mean         98.800228
std          14.607399
min           0.000000
25%          98.100000
50%          98.600000
75%          99.300000
max        9905.000000
Name: value, dtype: float64

After conversion - Temperature (now °C) describe:
count    738677.000000
mean         37.111238
std           8.115222
min         -17.777778
25%          36.722222
50%          37.000000
75%          37.388889
max        5485.000000
Name: value, dtype: float64

Weight lbs rows: 19365
Height inch rows: 101

Unit conversion done. Shape: (17408372, 8)


## STEP 3: 라벨 통합

In [ ]:
# 원본 라벨 기준으로 조건 체크 (순서 무관하게 안전)
L = all_events['label'].copy()


# ========== 의식/신경 ==========
all_events.loc[L.str.contains('Richmond-RAS', case=False, na=False), 'label'] = 'RASS'


# ========== 신체측정 ==========
all_events.loc[L.str.contains('weight', case=False, na=False), 'label'] = 'Weight'
all_events.loc[L.str.contains('height|length', case=False, na=False), 'label'] = 'Height'

# ========== 활력징후 ==========
all_events.loc[L.isin(['Heart Rate', 'Heart rate']), 'label'] = 'Heart Rate'
all_events.loc[L.str.contains('temperature', case=False, na=False), 'label'] = 'Temperature'
all_events.loc[L.str.contains('o2 sat', case=False, na=False), 'label'] = 'Oxygen Saturation'
all_events.loc[L.str.contains('blood pressure mean', case=False, na=False), 'label'] = 'Mean BP'

# ========== Ventilator ==========
all_events.loc[L.str.contains('ventilator', case=False, na=False), 'label'] = 'Ventilator'

# ============ 검사 ============
all_events.loc[L.str.contains('glucose', case=False, na=False), 'label'] = 'Glucose'
all_events.loc[L.str.contains('wbc|white blood', case=False, na=False), 'label'] = 'WBC'
all_events.loc[L.str.contains('hemoglobin', case=False, na=False), 'label'] = 'Hemoglobin'
all_events.loc[L.str.contains('hematocrit', case=False, na=False), 'label'] = 'Hematocrit'
all_events.loc[L.str.contains('platelet', case=False, na=False), 'label'] = 'Platelets'

all_events.loc[L.str.contains('sodium', case=False, na=False) & ~L.str.contains('bicarb', case=False, na=False), 'label'] = 'Sodium'
all_events.loc[L.str.contains('potassium', case=False, na=False), 'label'] = 'Potassium'
all_events.loc[L.str.contains('chloride', case=False, na=False), 'label'] = 'Chloride'
all_events.loc[L.str.contains('bicarbonate|hco3', case=False, na=False), 'label'] = 'Bicarbonate'
all_events.loc[L.str.contains('calcium', case=False, na=False), 'label'] = 'Calcium'
all_events.loc[L.str.contains('magnesium', case=False, na=False), 'label'] = 'Magnesium'
all_events.loc[L.str.contains('phosph', case=False, na=False), 'label'] = 'Phosphate'

all_events.loc[L.str.contains('urea nitrogen', case=False, na=False), 'label'] = 'BUN'
all_events.loc[L.str.contains('urine creatinine', case=False, na=False), 'label'] = 'Urine Creatinine'
all_events.loc[L.str.contains('creatine kinase', case=False, na=False), 'label'] = 'Creatine Kinase'
all_events.loc[L.str.contains('creatinine', case=False, na=False), 'label'] = 'Creatinine'
all_events.loc[L.str.contains('alt|alanine amino', case=False, na=False), 'label'] = 'ALT'
all_events.loc[L.str.contains('ast|aspartate amino', case=False, na=False), 'label'] = 'AST'
all_events.loc[L.str.contains('bilirubin', case=False, na=False), 'label'] = 'Bilirubin'
all_events.loc[L.str.contains('albumin', case=False, na=False), 'label'] = 'Albumin'

all_events.loc[L.str.contains('lactate', case=False, na=False), 'label'] = 'Lactate'
all_events.loc[L.str.contains('ammonia', case=False, na=False), 'label'] = 'Ammonia'
all_events.loc[L == 'pH', 'label'] = 'pH'
all_events.loc[L.str.contains('pco2', case=False, na=False), 'label'] = 'pCO2'
all_events.loc[L.str.contains('po2', case=False, na=False), 'label'] = 'pO2'
all_events.loc[L.str.contains('inr', case=False, na=False), 'label'] = 'INR'
all_events.loc[(L == 'PT') | L.str.startswith('PT ', na=False), 'label'] = 'PT'
all_events.loc[L.str.contains('ptt', case=False, na=False), 'label'] = 'PTT'

# ========== 약물 - 진정/진통/향정 ==========
all_events.loc[L.str.contains('propofol', case=False, na=False), 'label'] = 'Propofol'
all_events.loc[L.isin(['Fentanyl', 'Fentanyl (Concentrate)', 'Morphine Sulfate']), 'label'] = 'Opiates'
all_events.loc[L.isin(['Midazolam (Versed)', 'Lorazepam (Ativan)', 'Diazepam (Valium)']), 'label'] = 'Benzodiazepines'
all_events.loc[L.str.contains('ketamine', case=False, na=False), 'label'] = 'Ketamine'

# ========== 약물 - 승압제 ==========
all_events.loc[L.isin(['Vasopressin', 'Dobutamine', 'Phenylephrine (50/250)', 'Phenylephrine (200/250)']), 'label'] = 'Vasopressors'


print(f"Unified labels: {all_events['label'].nunique()}")
print(all_events['label'].value_counts())

In [114]:
# NULL 값 제거
all_events = all_events[all_events['value'].notna()]

print(f"Unified shape: {all_events.shape}")

Unified shape: (17400216, 8)


In [115]:
all_events.to_csv(OUTPUT_DIR / 'all_events_filtered.csv', index=False)

## STEP 4: 시간 계산 (ICU 입실 기준)

In [21]:
all_events = pd.read_csv(OUTPUT_DIR / 'all_events_filtered.csv')
adm_pat_icu = pd.read_csv(OUTPUT_DIR / 'adm_pat_icu_8hrs.csv')

C:\Users\김한재\AppData\Local\Temp\ipykernel_9860\356639417.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  all_events = pd.read_csv(OUTPUT_DIR / 'all_events_filtered.csv')


In [22]:
# adm_pat_icu와 조인
all_events = all_events.merge(
    adm_pat_icu[['stay_id', 'age', 'gender', 'los', 'admission_type', 'intime']],
    on='stay_id',
    how='inner'
)

# 시간 계산 (ICU 입실 기준, 시간 단위)
all_events['charttime'] = pd.to_datetime(all_events['charttime'])
all_events['intime'] = pd.to_datetime(all_events['intime'])
all_events['hours'] = (
    (all_events['charttime'] - all_events['intime'])
    .dt.total_seconds() / 3600.0
)

# ICU 입실 이후 데이터만 유지
all_events = all_events[all_events['hours'] >= 0]

# 60분 비닝
all_events['bin'] = all_events['hours'].astype(int)

print(f"With time shape: {all_events.shape}")
print(f"Unique stay_ids: {all_events['stay_id'].nunique()}")
print(f"Hours range: {all_events['hours'].min():.2f} ~ {all_events['hours'].max():.2f}")

With time shape: (17400216, 15)
Unique stay_ids: 7661
Hours range: 0.00 ~ 5432.36


## STEP 5: 60분 비닝 및 피봇 (최종 시계열)

In [23]:
# 피봇할 컬럼 목록 정의 (통합된 라벨명 기준)
PIVOT_COLUMNS = [
    # 활력징후
    'Heart Rate', 'Temperature', 'Oxygen Saturation', 'Mean BP',

    # 의식/신경
    'RASS',

    # 신체 측정
    'Weight', 'Height',

    # 검사 - 혈액
    'WBC', 'Hemoglobin', 'Hematocrit', 'Platelets',

    # 검사 - 전해질
    'Sodium', 'Potassium', 'Chloride', 'Bicarbonate',
    'Calcium', 'Magnesium', 'Phosphate',

    # 검사 - 신장/간
    'BUN', 'Creatinine', 'Creatine Kinase',
    'ALT', 'AST', 'Bilirubin', 'Albumin',

    # 검사 - 기타
    'Glucose', 'Lactate', 'Ammonia', 'pH', 'pCO2', 'pO2',

    # 약물 - 진정제/진통제 (통합 라벨)
    'Propofol', 'Opiates', 'Benzodiazepines', 'Ketamine',

    # 약물 - 승압제 (통합 라벨)
    'Vasopressors',

    # 기타
    'Ventilator',

    # 섬망 평가 (CAM-ICU 하위 항목)
    'CAM-ICU MS Change', 'CAM-ICU Inattention',
    'CAM-ICU Disorganized thinking', 'CAM-ICU RASS LOC',
]

In [24]:
# 기본 정보 집계
base_info = all_events.groupby(['stay_id', 'bin']).agg({
    'hours': 'max',
    'age': 'max',
    'gender': 'first',
    'los': 'max'
}).reset_index()

print(f"Base info shape: {base_info.shape}")

Base info shape: (2868290, 6)


In [25]:
base_info.head()

,stay_id,bin,hours,age,gender,los
0,30004018,0,0.966667,56,1,16.094329
1,30004018,1,1.966667,56,1,16.094329
2,30004018,2,2.483333,56,1,16.094329
3,30004018,3,3.966667,56,1,16.094329
4,30004018,4,4.466667,56,1,16.094329


In [26]:
# 피봇 테이블 생성
# 데이터에 존재하는 라벨만 필터링
available_labels = all_events['label'].unique()
labels_to_pivot = [col for col in PIVOT_COLUMNS if col in available_labels]

print(f"Labels to pivot: {len(labels_to_pivot)}")
print(f"Available in data: {labels_to_pivot}")

# 피봇할 데이터 필터링
pivot_data = all_events[all_events['label'].isin(labels_to_pivot)]

Labels to pivot: 40
Available in data: ['Heart Rate', 'Temperature', 'Oxygen Saturation', 'Mean BP', 'RASS', 'Weight', 'Height', 'WBC', 'Hemoglobin', 'Hematocrit', 'Platelets', 'Sodium', 'Potassium', 'Chloride', 'Bicarbonate', 'Calcium', 'Magnesium', 'Phosphate', 'BUN', 'Creatinine', 'ALT', 'AST', 'Bilirubin', 'Albumin', 'Glucose', 'Lactate', 'Ammonia', 'pH', 'pCO2', 'pO2', 'Propofol', 'Opiates', 'Benzodiazepines', 'Ketamine', 'Vasopressors', 'Ventilator', 'CAM-ICU MS Change', 'CAM-ICU Inattention', 'CAM-ICU Disorganized thinking', 'CAM-ICU RASS LOC']


In [27]:
# 피봇 테이블 생성 (각 bin에서 MAX 값 사용)
pivoted = pivot_data.pivot_table(
    index=['stay_id', 'bin'],
    columns='label',
    values='value',
    aggfunc='max'
).reset_index()

print(f"Pivoted shape: {pivoted.shape}")

Pivoted shape: (2868268, 42)


In [28]:
# 기본 정보와 피봇 테이블 병합
timeseries = base_info.merge(pivoted, on=['stay_id', 'bin'], how='left')

# 정렬
timeseries = timeseries.sort_values(['stay_id', 'bin']).reset_index(drop=True)

print(f"\n=== Final Timeseries ===")
print(f"Shape: {timeseries.shape}")
print(f"Unique stay_ids: {timeseries['stay_id'].nunique()}")
print(f"Total rows: {len(timeseries)}")
print(f"\nColumns: {timeseries.columns.tolist()}")


=== Final Timeseries ===
Shape: (2868290, 46)
Unique stay_ids: 7661
Total rows: 2868290

Columns: ['stay_id', 'bin', 'hours', 'age', 'gender', 'los', 'ALT', 'AST', 'Albumin', 'Ammonia', 'BUN', 'Benzodiazepines', 'Bicarbonate', 'Bilirubin', 'CAM-ICU Disorganized thinking', 'CAM-ICU Inattention', 'CAM-ICU MS Change', 'CAM-ICU RASS LOC', 'Calcium', 'Chloride', 'Creatinine', 'Glucose', 'Heart Rate', 'Height', 'Hematocrit', 'Hemoglobin', 'Ketamine', 'Lactate', 'Magnesium', 'Mean BP', 'Opiates', 'Oxygen Saturation', 'Phosphate', 'Platelets', 'Potassium', 'Propofol', 'RASS', 'Sodium', 'Temperature', 'Vasopressors', 'Ventilator', 'WBC', 'Weight', 'pCO2', 'pH', 'pO2']


In [29]:
timeseries.head(10)

,stay_id,bin,hours,age,gender,los,ALT,AST,Albumin,Ammonia,...,RASS,Sodium,Temperature,Vasopressors,Ventilator,WBC,Weight,pCO2,pH,pO2
0,30004018,0,0.966667,56,1,16.094329,NaN,NaN,NaN,NaN,...,NaN,NaN,37.000000,NaN,1.0,NaN,NaN,NaN,NaN,NaN
1,30004018,1,1.966667,56,1,16.094329,NaN,NaN,NaN,NaN,...,1.0,126.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,30004018,2,2.483333,56,1,16.094329,NaN,NaN,NaN,NaN,...,NaN,NaN,36.888889,NaN,1.0,NaN,NaN,NaN,NaN,NaN
3,30004018,3,3.966667,56,1,16.094329,NaN,NaN,NaN,NaN,...,NaN,57.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,30004018,4,4.466667,56,1,16.094329,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,30004018,5,5.966667,56,1,16.094329,71.0,58.0,NaN,NaN,...,NaN,126.0,NaN,NaN,NaN,39.3,NaN,NaN,NaN,NaN
6,30004018,6,6.466667,56,1,16.094329,NaN,NaN,NaN,NaN,...,NaN,NaN,36.833333,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,30004018,7,7.466667,56,1,16.094329,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
8,30004018,8,8.466667,56,1,16.094329,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,30004018,9,9.466667,56,1,16.094329,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN


## STEP 6: CAM-ICU 양성/음성 분류

In [30]:
# CAM-ICU 양성 판정 기준
# CAM Positive = (Feature1 AND Feature2) AND (Feature3 OR Feature4)
#   Feature1: CAM-ICU MS Change == 1 (의식 변화)
#   Feature2: CAM-ICU Inattention == 1 or 4 (주의력 장애)
#   Feature3: CAM-ICU RASS LOC == 1 (의식 수준 변화, = Altered LOC)
#   Feature4: CAM-ICU Disorganized thinking == 1 (비조직적 사고)

# 실제 CAM-ICU 평가가 수행된 시점 판별: 4개 하위 항목 중 하나라도 값이 있으면 평가 수행
cam_sub_cols = ['CAM-ICU MS Change', 'CAM-ICU Inattention',
                'CAM-ICU Disorganized thinking', 'CAM-ICU RASS LOC']
has_cam_assessment = timeseries[cam_sub_cols].notna().any(axis=1)

# 양성 판정
feature1_pos = timeseries['CAM-ICU MS Change'] == 1
feature2_pos = (timeseries['CAM-ICU Inattention'] == 1) | (timeseries['CAM-ICU Inattention'] == 4)
feature3_pos = timeseries['CAM-ICU RASS LOC'] == 1
feature4_pos = timeseries['CAM-ICU Disorganized thinking'] == 1

cam_positive = (feature1_pos & feature2_pos) & (feature3_pos | feature4_pos)

# CAM-ICU 컬럼 생성: 평가 있는 시점만 0/1, 나머지는 NaN
timeseries['CAM-ICU'] = np.nan
timeseries.loc[has_cam_assessment, 'CAM-ICU'] = 0  # 평가 있으나 음성
timeseries.loc[cam_positive, 'CAM-ICU'] = 1         # 양성

print(f"Feature1 (MS Change) positive: {feature1_pos.sum()}")
print(f"Feature2 (Inattention) positive: {feature2_pos.sum()}")
print(f"Feature3 (RASS LOC) positive: {feature3_pos.sum()}")
print(f"Feature4 (Disorganized thinking) positive: {feature4_pos.sum()}")
print(f"\n=== CAM-ICU 평가 현황 ===")
print(f"Total records: {len(timeseries):,}")
print(f"CAM-ICU 평가 수행된 시점: {has_cam_assessment.sum():,}")
print(f"  - Positive (1): {(timeseries['CAM-ICU'] == 1).sum():,}")
print(f"  - Negative (0): {(timeseries['CAM-ICU'] == 0).sum():,}")
print(f"CAM-ICU 평가 없는 시점 (NaN): {timeseries['CAM-ICU'].isna().sum():,}")
print(f"\n양성 기록이 있는 stay 수: {timeseries[timeseries['CAM-ICU'] == 1]['stay_id'].nunique()}")
print(f"음성만 있는 stay 수: {timeseries[timeseries['CAM-ICU'] == 0]['stay_id'].nunique() - timeseries[timeseries['CAM-ICU'] == 1]['stay_id'].nunique()}")

Feature1 (MS Change) positive: 95183
Feature2 (Inattention) positive: 74752
Feature3 (RASS LOC) positive: 355
Feature4 (Disorganized thinking) positive: 7017

=== CAM-ICU 평가 현황 ===
Total records: 2,868,290
CAM-ICU 평가 수행된 시점: 200,909
  - Positive (1): 6,767
  - Negative (0): 194,142
CAM-ICU 평가 없는 시점 (NaN): 2,667,381

양성 기록이 있는 stay 수: 2471
음성만 있는 stay 수: 5185


In [32]:
# CAM-ICU 하위 항목 제거 (라벨링 완료 후 불필요)
cam_sub_cols = ['CAM-ICU MS Change', 'CAM-ICU Inattention',
                'CAM-ICU Disorganized thinking', 'CAM-ICU RASS LOC']
timeseries.drop(columns=cam_sub_cols, inplace=True)

print(f"Final columns ({len(timeseries.columns)}): {timeseries.columns.tolist()}")
print(f"\nCAM-ICU value counts (NaN excluded):")
print(timeseries['CAM-ICU'].value_counts())
print(f"NaN count: {timeseries['CAM-ICU'].isna().sum():,}")

Final columns (43): ['stay_id', 'bin', 'hours', 'age', 'gender', 'los', 'ALT', 'AST', 'Albumin', 'Ammonia', 'BUN', 'Benzodiazepines', 'Bicarbonate', 'Bilirubin', 'Calcium', 'Chloride', 'Creatinine', 'Glucose', 'Heart Rate', 'Height', 'Hematocrit', 'Hemoglobin', 'Ketamine', 'Lactate', 'Magnesium', 'Mean BP', 'Opiates', 'Oxygen Saturation', 'Phosphate', 'Platelets', 'Potassium', 'Propofol', 'RASS', 'Sodium', 'Temperature', 'Vasopressors', 'Ventilator', 'WBC', 'Weight', 'pCO2', 'pH', 'pO2', 'CAM-ICU']

CAM-ICU value counts (NaN excluded):
CAM-ICU
0.0    194142
1.0      6767
Name: count, dtype: int64
NaN count: 2,667,381


In [35]:
# 전체 timeseries 저장 (CAM-ICU: 평가 시점만 0/1, 나머지 NaN)
timeseries.to_csv(OUTPUT_DIR / 'all_timeseries.csv', index=False)
print(f"Saved: all_timeseries.csv ({len(timeseries):,} rows, {timeseries['stay_id'].nunique()} stays)")

Saved: all_timeseries.csv (2,868,290 rows, 7661 stays)


## STEP 7: Imputation 

In [13]:
timeseries = pd.read_csv(OUTPUT_DIR / 'all_timeseries.csv')

In [36]:
# ========================================================================
# 7-1. Zero-fill: 약물 및 장비 컬럼
# ========================================================================
# 약물/장비는 기록이 없으면 투여/사용하지 않은 것이므로 0으로 채움

zero_fill_cols = ['Propofol', 'Opiates', 'Benzodiazepines', 'Ketamine',
                  'Vasopressors', 'Ventilator']

for col in zero_fill_cols:
    if col in timeseries.columns:
        before_null = timeseries[col].isnull().sum()
        timeseries[col] = timeseries[col].fillna(0)
        print(f"  {col}: {before_null:,} NaN → 0  (filled {before_null:,})")

print(f"\n7-1 완료: 약물/장비 컬럼 zero-fill")

  Propofol: 2,692,229 NaN → 0  (filled 2,692,229)
  Opiates: 2,716,868 NaN → 0  (filled 2,716,868)
  Benzodiazepines: 2,812,721 NaN → 0  (filled 2,812,721)
  Ketamine: 2,858,944 NaN → 0  (filled 2,858,944)
  Vasopressors: 2,819,988 NaN → 0  (filled 2,819,988)
  Ventilator: 2,450,214 NaN → 0  (filled 2,450,214)

7-1 완료: 약물/장비 컬럼 zero-fill


In [38]:
# ========================================================================
# 7-2. Patient-wise forward-fill + backward-fill: Weight, Height
# ========================================================================
# 체중/신장은 환자별로 잘 변하지 않는 값이므로,
# 같은 환자 내에서 시간순으로 forward-fill → backward-fill 수행

static_fill_cols = ['Weight', 'Height']

for col in static_fill_cols:
    if col in timeseries.columns:
        before_null = timeseries[col].isnull().sum()
        timeseries[col] = timeseries.groupby('stay_id')[col].transform(lambda v: v.ffill())
        timeseries[col] = timeseries.groupby('stay_id')[col].transform(lambda v: v.bfill())
        after_null = timeseries[col].isnull().sum()
        print(f"  {col}: {before_null:,} → {after_null:,} NaN  (filled {before_null - after_null:,})")

print(f"\n7-2 완료: Weight, Height 환자별 ffill + bfill")

  Weight: 2,660,533 → 41,472 NaN  (filled 2,619,061)
  Height: 2,868,189 → 2,827,199 NaN  (filled 40,990)

7-2 완료: Weight, Height 환자별 ffill + bfill


In [39]:
# ========================================================================
# 7-3. Patient-wise Forward-fill: 임상 측정값
# ========================================================================
# 활력징후, 검사값 등은 시간에 따라 연속적으로 변하므로,
# 마지막 측정값이 다음 시간대까지 유지된다고 가정 (forward-fill)
# 환자(stay_id) 단위로 수행하여 다른 환자 데이터가 섞이지 않도록 함

forward_fill_cols = [
    'Heart Rate', 'Temperature', 'Oxygen Saturation', 'Mean BP',
    'RASS',
    'WBC', 'Hemoglobin', 'Hematocrit', 'Platelets',
    'Sodium', 'Potassium', 'Chloride', 'Bicarbonate',
    'Calcium', 'Magnesium', 'Phosphate',
    'BUN', 'Creatinine', 'ALT', 'AST', 'Bilirubin', 'Albumin',
    'Glucose', 'Lactate', 'Ammonia', 'pH', 'pCO2', 'pO2',
]

# 데이터에 존재하는 컬럼만 필터
forward_fill_cols = [c for c in forward_fill_cols if c in timeseries.columns]

print("7-3. Forward-fill (환자별):")
for col in forward_fill_cols:
    before_null = timeseries[col].isnull().sum()
    timeseries[col] = timeseries.groupby('stay_id')[col].transform(lambda v: v.ffill())
    after_null = timeseries[col].isnull().sum()
    filled = before_null - after_null
    if filled > 0:
        print(f"  {col}: {before_null:,} → {after_null:,} NaN  (filled {filled:,})")

print(f"\n7-3 완료: 임상 측정값 forward-fill")

7-3. Forward-fill (환자별):
  Heart Rate: 40,318 → 2,909 NaN  (filled 37,409)
  Temperature: 1,982,008 → 37,568 NaN  (filled 1,944,440)
  Oxygen Saturation: 78,657 → 2,683 NaN  (filled 75,974)
  Mean BP: 283,884 → 9,381 NaN  (filled 274,503)
  RASS: 2,216,387 → 15,965 NaN  (filled 2,200,422)
  WBC: 2,692,820 → 50,262 NaN  (filled 2,642,558)
  Hemoglobin: 2,675,395 → 47,759 NaN  (filled 2,627,636)
  Hematocrit: 2,680,454 → 49,732 NaN  (filled 2,630,722)
  Platelets: 2,690,053 → 50,199 NaN  (filled 2,639,854)
  Sodium: 2,625,135 → 43,875 NaN  (filled 2,581,260)
  Potassium: 2,603,041 → 42,987 NaN  (filled 2,560,054)
  Chloride: 2,639,564 → 44,519 NaN  (filled 2,595,045)
  Bicarbonate: 2,656,747 → 47,769 NaN  (filled 2,608,978)
  Calcium: 2,669,522 → 57,972 NaN  (filled 2,611,550)
  Magnesium: 2,661,199 → 53,889 NaN  (filled 2,607,310)
  Phosphate: 2,667,890 → 56,906 NaN  (filled 2,610,984)
  BUN: 2,660,490 → 47,697 NaN  (filled 2,612,793)
  Creatinine: 2,660,424 → 47,604 NaN  (filled 2,612,

In [40]:
# ========================================================================
# 7-4. Patient-wise Backward-fill: 임상 측정값
# ========================================================================
# Forward-fill로 채우지 못한 초기 시간대(첫 측정 이전 bin)의 결측을
# backward-fill로 보간 (가장 가까운 미래 측정값으로 채움)

backward_fill_cols = forward_fill_cols  # ffill과 동일한 컬럼 대상

print("7-4. Backward-fill (환자별):")
for col in backward_fill_cols:
    before_null = timeseries[col].isnull().sum()
    timeseries[col] = timeseries.groupby('stay_id')[col].transform(lambda v: v.bfill())
    after_null = timeseries[col].isnull().sum()
    filled = before_null - after_null
    if filled > 0:
        print(f"  {col}: {before_null:,} → {after_null:,} NaN  (filled {filled:,})")

print(f"\n7-4 완료: 임상 측정값 backward-fill")

7-4. Backward-fill (환자별):
  Heart Rate: 2,909 → 0 NaN  (filled 2,909)
  Temperature: 37,568 → 1,263 NaN  (filled 36,305)
  Oxygen Saturation: 2,683 → 0 NaN  (filled 2,683)
  Mean BP: 9,381 → 217 NaN  (filled 9,164)
  RASS: 15,965 → 0 NaN  (filled 15,965)
  WBC: 50,262 → 11,479 NaN  (filled 38,783)
  Hemoglobin: 47,759 → 11,479 NaN  (filled 36,280)
  Hematocrit: 49,732 → 11,479 NaN  (filled 38,253)
  Platelets: 50,199 → 11,479 NaN  (filled 38,720)
  Sodium: 43,875 → 11,479 NaN  (filled 32,396)
  Potassium: 42,987 → 11,479 NaN  (filled 31,508)
  Chloride: 44,519 → 11,479 NaN  (filled 33,040)
  Bicarbonate: 47,769 → 11,479 NaN  (filled 36,290)
  Calcium: 57,972 → 11,479 NaN  (filled 46,493)
  Magnesium: 53,889 → 11,479 NaN  (filled 42,410)
  Phosphate: 56,906 → 11,479 NaN  (filled 45,427)
  BUN: 47,697 → 11,479 NaN  (filled 36,218)
  Creatinine: 47,604 → 11,479 NaN  (filled 36,125)
  ALT: 255,525 → 92,516 NaN  (filled 163,009)
  AST: 552,075 → 323,961 NaN  (filled 228,114)
  Bilirubin: 58

In [41]:
# ========================================================================
# 7-5. Imputation 후 결측률 확인
# ========================================================================
# 환자별(patient-wise) 기준으로 결측률을 체크
# - 환자 내 한 번도 측정되지 않은 변수는 ffill/bfill로도 채울 수 없음
# - CAM-ICU는 평가 시점만 값이 있으므로 결측률 체크에서 제외

feature_cols = [c for c in timeseries.columns if c not in ['stay_id', 'bin', 'hours', 'age', 'gender', 'los', 'CAM-ICU']]

percent_missing_after = timeseries[feature_cols].isnull().sum() * 100 / len(timeseries)
missing_df_after = pd.DataFrame({
    'column': feature_cols,
    'after_pct': percent_missing_after.values,
}).sort_values('after_pct', ascending=False).reset_index(drop=True)

print("=== Imputation 전후 결측률 비교 ===")
print(missing_df_after.to_string())

# CAM-ICU 별도 확인
cam_total = len(timeseries)
cam_assessed = timeseries['CAM-ICU'].notna().sum()
print(f"\n=== CAM-ICU (별도) ===")
print(f"평가 수행 시점: {cam_assessed:,} / {cam_total:,} ({cam_assessed/cam_total*100:.2f}%)")
print(f"미평가 시점 (NaN): {cam_total - cam_assessed:,} ({(cam_total-cam_assessed)/cam_total*100:.2f}%)")

=== Imputation 전후 결측률 비교 ===
               column  after_pct
0              Height  98.567404
1             Ammonia  91.214347
2             Albumin  19.022309
3           Bilirubin  12.382883
4                 AST  11.294569
5             Lactate   7.860572
6                pCO2   5.203588
7                 pO2   5.194349
8                 ALT   3.225476
9                  pH   2.010187
10             Weight   1.445879
11                BUN   0.400204
12          Magnesium   0.400204
13         Hematocrit   0.400204
14         Hemoglobin   0.400204
15          Platelets   0.400204
16         Creatinine   0.400204
17           Chloride   0.400204
18        Bicarbonate   0.400204
19            Calcium   0.400204
20                WBC   0.400204
21          Phosphate   0.400204
22             Sodium   0.400204
23          Potassium   0.400204
24        Temperature   0.044033
25            Glucose   0.007810
26            Mean BP   0.007565
27    Benzodiazepines   0.000000
28           K

In [42]:
# Imputed 데이터 저장
timeseries.to_csv(OUTPUT_DIR / 'timeseries_imputed.csv', index=False)
print(f"Saved: timeseries_imputed.csv ({len(timeseries):,} rows, {timeseries['stay_id'].nunique()} stays)")

Saved: timeseries_imputed.csv (2,868,290 rows, 7661 stays)


## STEP 8: CAM-ICU 평가 시점 기준 8시간 윈도우 집계

각 CAM-ICU 평가 시점으로부터 **이전 8시간**(cam_bin-7 ~ cam_bin) 데이터를 하나의 row로 집계:
- **활력징후** (Heart Rate, Mean BP, Oxygen Saturation, Temperature): 8시간 mean + std
- **약물/장비** (Propofol, Opiates, Benzodiazepines, Ketamine, Vasopressors, Ventilator): 8시간 내 사용 이력 → 1/0
- **신체계측 + 검사결과** (나머지): 8시간 내 가장 최신 시점 값

In [55]:
timeseries = pd.read_csv(OUTPUT_DIR / 'timeseries_imputed.csv')

In [56]:
# CAM-ICU 평가가 존재하는 시점 식별
cam_assessments = timeseries[timeseries['CAM-ICU'].notna()][['stay_id', 'bin', 'CAM-ICU']].copy()
cam_assessments.rename(columns={'bin': 'cam_bin'}, inplace=True)

# 입실 후 8시간 미만 시점의 평가 제외 (8시간 윈도우가 온전하지 않으므로)
before_filter = len(cam_assessments)
cam_assessments = cam_assessments[cam_assessments['cam_bin'] >= 7]
print(f"cam_bin < 7 제외: {before_filter:,} → {len(cam_assessments):,} ({before_filter - len(cam_assessments):,}건 제외)")

print(f"\nCAM-ICU 평가 시점 수: {len(cam_assessments):,}")
print(f"  - Positive (1): {(cam_assessments['CAM-ICU'] == 1).sum():,}")
print(f"  - Negative (0): {(cam_assessments['CAM-ICU'] == 0).sum():,}")
print(f"  - Unique stays: {cam_assessments['stay_id'].nunique()}")

cam_bin < 7 제외: 200,909 → 195,440 (5,469건 제외)

CAM-ICU 평가 시점 수: 195,440
  - Positive (1): 6,701
  - Negative (0): 188,739
  - Unique stays: 7644


In [57]:
# 각 평가 시점에 대해 8시간 윈도우 bin 목록 생성 (cam_bin-7 ~ cam_bin)
cam_assessments['window_bins'] = cam_assessments['cam_bin'].apply(
    lambda b: list(range(max(0, int(b) - 7), int(b) + 1))
)
cam_exploded = cam_assessments.explode('window_bins')
cam_exploded['bin'] = cam_exploded['window_bins'].astype(int)
cam_exploded.drop(columns='window_bins', inplace=True)

print(f"Exploded shape (assessments × window bins): {cam_exploded.shape}")

# timeseries와 merge (CAM-ICU는 cam_assessments에서 이미 가져옴)
window_data = cam_exploded.merge(
    timeseries.drop(columns='CAM-ICU'),
    on=['stay_id', 'bin'],
    how='inner'
)
print(f"Window data shape: {window_data.shape}")

Exploded shape (assessments × window bins): (1563520, 4)
Window data shape: (1531897, 44)


In [ ]:
# 컬럼 분류
vital_mean_std_cols = ['Heart Rate', 'Mean BP', 'Oxygen Saturation', 'Temperature']

binary_cols = ['Propofol', 'Opiates', 'Benzodiazepines', 'Ketamine',
               'Vasopressors', 'Ventilator']

latest_value_cols = [
    # 신체계측
    'Weight', 'Height',
    # 의식
    'RASS',
    # 혈액
    'WBC', 'Hemoglobin', 'Hematocrit', 'Platelets',
    # 전해질
    'Sodium', 'Potassium', 'Chloride', 'Bicarbonate',
    'Calcium', 'Magnesium', 'Phosphate',
    # 신장/간
    'BUN', 'Creatinine', 'ALT', 'AST', 'Bilirubin', 'Albumin',
    # 기타 검사
    'Glucose', 'Lactate', 'Ammonia', 'pH', 'pCO2', 'pO2',
]

group_key = ['stay_id', 'cam_bin', 'CAM-ICU']

# ── 1. Demographics (첫 번째 값) ──
demo_agg = window_data.groupby(group_key).agg(
    age=('age', 'first'),
    gender=('gender', 'first'),
    los=('los', 'first'),
).reset_index()

# ── 2. 활력징후: 8시간 mean + std ──
vital_means = window_data.groupby(group_key)[vital_mean_std_cols].mean()
vital_means.columns = [f'{c}_mean' for c in vital_means.columns]
vital_stds = window_data.groupby(group_key)[vital_mean_std_cols].std()
vital_stds.columns = [f'{c}_std' for c in vital_stds.columns]
vital_agg = pd.concat([vital_means, vital_stds], axis=1).reset_index()

# ── 3. 약물/장비: 8시간 내 사용 이력 있으면 1, 없으면 0 ──
binary_agg = window_data.groupby(group_key)[binary_cols].max().reset_index()
for col in binary_cols:
    binary_agg[col] = (binary_agg[col] > 0).astype(int)

# ── 4. 신체계측 + 검사결과: 가장 최신 시점(bin이 큰 쪽)의 값 ──
window_sorted = window_data.sort_values('bin', ascending=False)
latest_agg = window_sorted.groupby(group_key)[latest_value_cols].first().reset_index()

# ── 병합 ──
final_df = demo_agg.merge(vital_agg, on=group_key)
final_df = final_df.merge(binary_agg, on=group_key)
final_df = final_df.merge(latest_agg, on=group_key)

final_df = final_df.sort_values(['stay_id', 'cam_bin']).reset_index(drop=True)

print(f"\n=== Final Dataset ===")
print(f"Shape: {final_df.shape}")
print(f"Unique stays: {final_df['stay_id'].nunique()}")
print(f"\nCAM-ICU distribution:")
print(final_df['CAM-ICU'].value_counts())
print(f"\nColumns ({len(final_df.columns)}):")
print(final_df.columns.tolist())

In [62]:
# 결과 확인
print("=== 결측률 ===")
missing_pct = final_df.isnull().sum() * 100 / len(final_df)
print(missing_pct[missing_pct > 0].sort_values(ascending=False).to_string())
if missing_pct.sum() == 0:
    print("결측 없음")

=== 결측률 ===
Height                   98.632317
Ammonia                  92.217560
Albumin                  20.563856
Bilirubin                13.634875
AST                      12.696991
Lactate                   8.729533
pCO2                      5.935837
pO2                       5.928674
ALT                       3.748977
pH                        2.092714
Weight                    1.746828
Hematocrit                0.324396
Hemoglobin                0.324396
WBC                       0.324396
Sodium                    0.324396
Platelets                 0.324396
Creatinine                0.324396
BUN                       0.324396
Phosphate                 0.324396
Calcium                   0.324396
Bicarbonate               0.324396
Chloride                  0.324396
Potassium                 0.324396
Magnesium                 0.324396
Temperature_std           0.038887
Mean BP_std               0.022513
Temperature_mean          0.020467
Heart Rate_std            0.018420
Oxygen S

In [63]:
# 결측률 높은 변수 제외 (Height, Ammonia, Albumin, Bilirubin, AST)
drop_cols = ['Height', 'Ammonia', 'Albumin', 'Bilirubin', 'AST']
final_df.drop(columns=drop_cols, inplace=True)
print(f"제외된 변수: {drop_cols}")
print(f"Shape after drop: {final_df.shape}")
print(f"Columns ({len(final_df.columns)}): {final_df.columns.tolist()}")

제외된 변수: ['Height', 'Ammonia', 'Albumin', 'Bilirubin', 'AST']
Shape after drop: (195440, 41)
Columns (41): ['stay_id', 'cam_bin', 'CAM-ICU', 'age', 'gender', 'los', 'Heart Rate_mean', 'Mean BP_mean', 'Oxygen Saturation_mean', 'Temperature_mean', 'Heart Rate_std', 'Mean BP_std', 'Oxygen Saturation_std', 'Temperature_std', 'Propofol', 'Opiates', 'Benzodiazepines', 'Ketamine', 'Vasopressors', 'Ventilator', 'Weight', 'RASS', 'WBC', 'Hemoglobin', 'Hematocrit', 'Platelets', 'Sodium', 'Potassium', 'Chloride', 'Bicarbonate', 'Calcium', 'Magnesium', 'Phosphate', 'BUN', 'Creatinine', 'ALT', 'Glucose', 'Lactate', 'pH', 'pCO2', 'pO2']


In [64]:
# 저장
final_df.to_csv(OUTPUT_DIR / 'final_dataset.csv', index=False)
print(f"Saved: final_dataset.csv ({len(final_df):,} rows, {final_df['stay_id'].nunique()} stays)")

Saved: final_dataset.csv (195,440 rows, 7644 stays)
